In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when 
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import roc_curve, auc as sk_auc
import matplotlib.pyplot as plt
import pandas as pd




spark = SparkSession.builder.getOrCreate()

df = spark.read\
      .option('header','true')\
      .option('inferSchema', 'true') \
      .csv('/Volumes/workspace/credit_risk/credit/credit_risk_dataset.csv')  
display(df)

In [0]:
df.printSchema()

In [0]:
print('Rows:', df.count())
print('Column:', len(df.columns))

In [0]:
df.groupBy('loan_status').count().show()

In [0]:
import pyspark.sql.functions as F
display(df.select([F.count(F.when(col(c).isNull(), c)).alias(c) for c in df.columns]))

In [0]:
df = df.dropna(subset=['loan_int_rate'])
display(df)

In [0]:
imputer = Imputer(inputCol= 'person_emp_length', outputCol= 'person_emp_length')
df = imputer.fit(df).transform(df)

In [0]:
numeric_cols = [
    'person_age',
    'person_income',
    'person_emp_length', 
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length'
    ]

categorical_cols =[
    'person_home_ownership',
    'loan_intent',
    'loan_grade',
    'cb_person_default_on_file'
    ]

df.select(numeric_cols + categorical_cols + ['loan_status']).printSchema()

In [0]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in numeric_cols + categorical_cols
]).show(truncate=False)

In [0]:
indexers = [
StringIndexer(inputCol= c, outputCol= c + '_index')
for c in categorical_cols
]

pipeline = Pipeline(stages = indexers)
df = pipeline.fit(df).transform(df)

df.select(categorical_cols + [c + '_index' for c in categorical_cols]).show(5)


In [0]:
features_cols = numeric_cols + [c + '_index' for c in categorical_cols]

assembler = VectorAssembler(
inputCols = features_cols,
outputCol= 'features'
)
df_ml = assembler.transform(df)

df_ml.select('features', 'loan_status').show(5, truncate= False)

In [0]:
train_df, test_df = df_ml.select('features', 'loan_status').randomSplit([0.7, 0.3], seed = 42)
print('Training set count:', train_df.count())
print('Testing set count:', test_df.count())

In [0]:
lr = LogisticRegression(
    featuresCol = 'features',
    labelCol= 'loan_status',
    maxIter= 10
)
lr_model = lr.fit(train_df)
lr_predictions = lr_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(
    labelCol = 'loan_status',
    rawPredictionCol= 'rawPrediction',
    metricName = 'areaUnderROC'
)
lr_auc = evaluator.evaluate(lr_predictions)
print('Logistic Regression AUC:', lr_auc)

lr_coefficient = pd.DataFrame(
    list(zip(features_cols, lr_model.coefficients)),
    columns = ['feature', "Coefficient"]
).sort_values(by ='Coefficient', key=abs, ascending=False)
display(lr_coefficient)

In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="loan_status",
    numTrees=100,
    maxDepth=5,   
    seed=42
)

rf_model = rf.fit(train_df)

rf_predictions = rf_model.transform(test_df)

rf_auc = evaluator.evaluate(rf_predictions)
print('Random Forest AUC:', rf_auc)

rf_feat_imp = pd.DataFrame(
    list(zip(features_cols, rf_model.featureImportances)),
    columns=["Feature", "Importance"]
).sort_values(by="Importance", ascending=False)

display(rf_feat_imp)



In [0]:
lr_pred_pd = lr_predictions.select('loan_status', 'probability').toPandas()

y_true = lr_pred_pd['loan_status']
y_scores =lr_pred_pd['probability'].apply(lambda x: float(x[1]))
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
lr_roc_auc = sk_auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color ='blue', lw= 2, label ='ROC curve (AUC = %0.3f)'% lr_roc_auc)
plt.plot([0, 1], [0, 1], color ='red', lw= 1, linestyle= '--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Reciever Operating Charateristic (ROC)')
plt.legend(loc='lower right')
plt.show()



In [0]:
rf_preds_pd = rf_predictions.select("loan_status", "probability").toPandas()
y_scores_rf = rf_preds_pd["probability"].apply(lambda x: float(x[1]))
fpr_rf, tpr_rf, _ = roc_curve(y_true, y_scores_rf)
roc_auc_rf = sk_auc(fpr_rf, tpr_rf)
plt.plot(fpr_rf, tpr_rf, color='green', lw=2, label='RF ROC curve (AUC = %0.3f)' % roc_auc_rf)

plt.plot([0, 1], [0, 1], color='red', lw=1, linestyle='--')
plt.xlim([0,1])
plt.ylim([0,1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc="lower right")
plt.show()